# 15 - Japanese CLIP (Rinna) モデル評価

> **⚠️ スキップ**: このノートブックは評価対象外です。`japanese_clip` ライブラリの依存関係が Python 3.14 に対応しておらず、インストールできないため評価をスキップしました。

## 概要
Rinna の Japanese CLIP モデルを評価する。

## モデル情報
- **Model**: rinna/japanese-clip-vit-b-16
- **Embedding dimension**: 512
- **Type**: Multimodal (Image + Japanese Text)
- **特徴**: 日本語テキストをネイティブサポート

In [2]:
import time
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm

from image_vector_poc import JapaneseCLIPEmbedder
from image_vector_poc.evaluation import EvaluationReporter, evaluate_embeddings

## 設定

In [3]:
DB_PATH = Path("../data/images.duckdb")
OUTPUT_DIR = Path("../data/evaluations")
BATCH_SIZE = 32
RANDOM_STATE = 42

## 画像カタログの読み込み

In [4]:
conn = duckdb.connect(str(DB_PATH), read_only=True)
query = """
    SELECT id, file_path, category, file_name
    FROM image_catalog
    ORDER BY category, file_name
"""
catalog = conn.execute(query).fetchall()
conn.close()

image_ids = [r[0] for r in catalog]
file_paths = [r[1] for r in catalog]
categories = [r[2] for r in catalog]

category_labels = np.array(categories)
category_counts = pd.Series(categories).value_counts().to_dict()
unique_categories = list(category_counts.keys())

print(f"Total images: {len(catalog)}")
print(f"Categories: {unique_categories}")

Total images: 385
Categories: ['EuroPython2025', 'PyConJP2025', 'PyConJP2025-PreCampHiroshima', 'KashiwaVillagePark2026', 'TokyoNight202505', 'terada']


## モデルの初期化

In [5]:
print("Loading Japanese CLIP (Rinna) model...")
embedder = JapaneseCLIPEmbedder(device="cuda")
print(f"Model: {embedder.model_name}")
print(f"Embedding dimension: {embedder.embedding_dim}")
print(f"Device: {embedder.device}")

Loading Japanese CLIP (Rinna) model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/787M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/3 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: rinna/japanese-clip-vit-b-16
Key                                                                  | Status     | 
---------------------------------------------------------------------+------------+-
text_model.encoder.layer.{0...11}.attention.output.LayerNorm.weight  | UNEXPECTED | 
vision_model.encoder.layer.{0...11}.attention.attention.query.bias   | UNEXPECTED | 
vision_model.encoder.layer.{0...11}.attention.output.dense.bias      | UNEXPECTED | 
text_model.encoder.layer.{0...11}.intermediate.dense.weight          | UNEXPECTED | 
text_model.encoder.layer.{0...11}.attention.self.value.weight        | UNEXPECTED | 
text_model.embeddings.position_embeddings.weight                     | UNEXPECTED | 
vision_model.encoder.layer.{0...11}.attention.attention.value.weight | UNEXPECTED | 
vision_model.encoder.layer.{0...11}.attention.output.dense.weight    | UNEXPECTED | 
vision_model.encoder.layer.{0...11}.layernorm_before.weight          | UNEXPECTED | 
text_mod

OSError: Can't load image processor for 'rinna/japanese-clip-vit-b-16'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'rinna/japanese-clip-vit-b-16' is the correct path to a directory containing a preprocessor_config.json file

## 画像のベクトル化

In [ ]:
print(f"Generating embeddings for {len(file_paths)} images...")

start_time = time.time()
embeddings_list = []

for i in tqdm(range(0, len(file_paths), BATCH_SIZE), desc="Embedding"):
    batch_paths = file_paths[i:i + BATCH_SIZE]
    batch_images = []
    for path in batch_paths:
        try:
            img = Image.open(path).convert("RGB")
            batch_images.append(img)
        except Exception as e:
            print(f"Error loading {path}: {e}")
            batch_images.append(Image.new("RGB", (224, 224), color="gray"))
    
    batch_embs = embedder.embed_images(batch_images)
    embeddings_list.append(batch_embs)

embeddings = np.vstack(embeddings_list)
processing_time = time.time() - start_time

print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"Processing time: {processing_time:.2f}s")
print(f"Speed: {len(file_paths) / processing_time:.1f} images/sec")

## 評価の実行

In [ ]:
print("Running evaluation...\n")

metrics = evaluate_embeddings(
    embeddings=embeddings,
    labels=category_labels,
    model_name=embedder.model_name,
    embedding_dim=embedder.embedding_dim,
    categories=unique_categories,
    category_counts=category_counts,
    processing_time=processing_time,
    random_state=RANDOM_STATE,
)

## 結果の表示

In [ ]:
print("=" * 60)
print("Evaluation Results - Japanese CLIP (Rinna)")
print("=" * 60)
print(f"Model: {metrics.model_name}")
print(f"Embedding dimension: {metrics.embedding_dim}")
print(f"Processing time: {metrics.processing_time_seconds:.2f}s")

print("\n--- t-SNE Metrics ---")
print(f"2D: Silhouette={metrics.silhouette_2d:.4f}, Trust={metrics.trustworthiness_2d:.4f}, DistRatio={metrics.distance_ratio_2d:.4f}")
print(f"3D: Silhouette={metrics.silhouette_3d:.4f}, Trust={metrics.trustworthiness_3d:.4f}, DistRatio={metrics.distance_ratio_3d:.4f}")

print("\n--- PCA Metrics ---")
print(f"2D: Silhouette={metrics.pca_silhouette_2d:.4f}, Variance={metrics.pca_variance_ratio_2d:.4f}")
print(f"3D: Silhouette={metrics.pca_silhouette_3d:.4f}, Variance={metrics.pca_variance_ratio_3d:.4f}")

## 日本語テキスト埋め込みのテスト

In [ ]:
# 日本語テキストのテスト
test_queries = [
    "青い空",
    "猫の写真",
    "美しい風景",
    "カンファレンス",
]

print("Japanese text embedding test:")
for query in test_queries:
    text_emb = embedder.embed_text(query)
    print(f"  '{query}': shape={text_emb.shape}, norm={np.linalg.norm(text_emb):.4f}")

## 結果の保存

In [ ]:
reporter = EvaluationReporter(OUTPUT_DIR)
filepath = reporter.save(metrics)
print(f"Results saved to: {filepath}")

## GPUメモリのクリーンアップ

In [ ]:
del embedder
del embeddings

import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU memory cleared.")